# 决策树回归：用分段常数逼近任意函数
> 难度标签：初级 | 预计时长：15-25分钟 | 前置知识：无学习经验


> 所属模块：02_监督学习/回归 | 源文件：决策树_回归.py | 核心功能：树形回归、max_depth 过拟合控制、GridSearchCV 调参

## 概述

决策树回归的思想：把特征空间递归地划分为若干矩形区域，每个区域的预测值等于该区域内训练样本目标值的均值。本质上是一个**分段常数函数**——树越深，分段越细，拟合越精确（但也越容易过拟合）。

脚本演示了决策树回归的训练、树结构展示、剪枝控制和超参数搜索。

**导入必要的库**

In [ ]:
# 确保中文输出正常（Windows 环境兼容）
import sys
try:
    sys.stdout.reconfigure(encoding='utf-8')
except AttributeError:
    pass

import numpy as np
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeRegressor, export_text
from sklearn.metrics import mean_squared_error, r2_score

## 数学原理

### 1. 决策树回归的叶节点输出

**代码对应**：`DecisionTreeRegressor(random_state=42).fit(X_train, y_train)` 训练回归树。

决策树回归将特征空间划分为 $M$ 个不相交的区域 $R_1, R_2, \ldots, R_M$，每个区域的预测值为该区域内训练样本目标值的均值：

$$f(\mathbf{x}) = \sum_{m=1}^{M} c_m \cdot \mathbb{I}(\mathbf{x} \in R_m), \quad c_m = \frac{1}{|R_m|}\sum_{\mathbf{x}_i \in R_m} y_i$$

### 2. 分裂准则：最小化 MSE

**代码对应**：sklearn 默认使用 `criterion="squared_error"`（MSE）。

在每个节点，选择特征 $j$ 和阈值 $s$ 使得分裂后的总 MSE 最小：

$$\min_{j, s} \left[\min_{c_1} \sum_{\mathbf{x}_i \in R_1(j,s)} (y_i - c_1)^2 + \min_{c_2} \sum_{\mathbf{x}_i \in R_2(j,s)} (y_i - c_2)^2\right]$$

其中 $R_1(j,s) = \{\mathbf{x} | x_j \leq s\}$，$R_2(j,s) = \{\mathbf{x} | x_j > s\}$。

最优 $c_1 = \bar{y}_{R_1}$，$c_2 = \bar{y}_{R_2}$（区域内均值）。

**信息增益（MSE 减少量）**：

$$\Delta \text{MSE} = \text{MSE}_{\text{parent}} - \left(\frac{n_{\text{left}}}{n}\text{MSE}_{\text{left}} + \frac{n_{\text{right}}}{n}\text{MSE}_{\text{right}}\right)$$

### 3. 特征重要性

**代码对应**：`dt.feature_importances_` 返回各特征的重要性。

特征 $j$ 的重要性由其在所有节点中带来的 MSE 减少量加权求和：

$$\text{Imp}(j) = \sum_{t: \text{node } t \text{ splits on } j} \frac{n_t}{N} \Delta\text{MSE}_t$$

其中 $n_t$ 为节点 $t$ 的样本数，$N$ 为总样本数。最后归一化使总和为 1。

### 4. 过拟合与剪枝

**代码对应**：代码中 `for depth in [2, 3, 5, 8, 10, 15, None]` 展示了树深度的影响。

无限制的决策树会在每个叶节点只包含一个训练样本，此时训练 MSE = 0（完美拟合），但泛化能力极差。

控制过拟合的超参数：
- `max_depth`：最大深度（最常用）
- `min_samples_split`：分裂所需最少样本数
- `min_samples_leaf`：叶节点最少样本数
- `max_features`：每次分裂考虑的最大特征数

### 5. 决策树的偏差-方差特性

决策树是**高方差、低偏差**的模型：
- **低偏差**：足够深的树可以逼近任意复杂函数
- **高方差**：训练数据的微小变化可能导致完全不同的树结构

这是 Bagging 和随机森林能有效降低决策树方差的根本原因。

### 6. 分类与回归树（CART）算法

sklearn 使用 CART 算法构建决策树。CART 的特点：
- 每个节点只做二分裂（不是多分裂）
- 支持分类和回归
- 回归树使用 MSE 作为分裂准则
- 分类树使用基尼系数或信息增益

**时间复杂度**：构建一棵深度为 $d$ 的平衡树，每层需 $O(np\log n)$ 排序，总复杂度 $O(npd\log n)$。

### 1. 生成数据

生成合成数据集，用于演示算法的核心行为。

In [ ]:
X, y = make_regression(n_samples=300, n_features=5, n_informative=3, noise=10, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### 2. 基础决策树回归

在回归任务上拟合模型，观察预测值与真实值的偏差。

In [ ]:
dt = DecisionTreeRegressor(random_state=42)
dt.fit(X_train, y_train)

print("=== 决策树回归 ===")
print(f"训练 R²: {dt.score(X_train, y_train):.4f}")
print(f"测试 R²: {dt.score(X_test, y_test):.4f}")
print(f"树深度: {dt.get_depth()}")
print(f"叶节点数: {dt.get_n_leaves()}")
# --- 输出结果 ---
print(f"特征重要性: {dt.feature_importances_.round(4)}")

### 3. 文本树结构

运行 3. 文本树结构 的代码，观察算法在该环节的行为。

In [ ]:
print("\n=== 树结构（前 20 行）===")
tree_rules = export_text(dt, feature_names=[f"X{i}" for i in range(5)], decimals=2)
print("\n".join(tree_rules.split("\n")[:20]))

### 4. max_depth 过拟合控制

运行 4. max_depth 过拟合控制 的代码，观察算法在该环节的行为。

In [ ]:
print("\n=== max_depth 影响 ===")
for depth in [2, 3, 5, 8, 10, 15, None]:
    dt_d = DecisionTreeRegressor(max_depth=depth, random_state=42)
    dt_d.fit(X_train, y_train)
    train_r2 = dt_d.score(X_train, y_train)
# --- 继续 ---
    test_r2 = dt_d.score(X_test, y_test)
    depth_str = str(depth) if depth else "无限制"
    print(f"  depth={depth_str:>3}: 训练R²={train_r2:.4f}, 测试R²={test_r2:.4f}, "
          f"叶节点={dt_d.get_n_leaves()}")

### 5. 超参数搜索

探索关键超参数对模型行为的影响。

In [ ]:
print("\n=== GridSearchCV ===")
param_grid = {
    "max_depth": [3, 5, 7, 10],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
# --- 继续 ---
    "max_features": ["sqrt", "log2", None],
}
gs = GridSearchCV(
    DecisionTreeRegressor(random_state=42),
    param_grid, cv=5, scoring="r2", n_jobs=-1
# --- 继续 ---
)
gs.fit(X_train, y_train)
print(f"最佳参数: {gs.best_params_}")
print(f"最佳 CV R²: {gs.best_score_:.4f}")
print(f"测试 R²: {gs.best_estimator_.score(X_test, y_test):.4f}")

### 6. 预测示例

使用训练好的模型进行预测，观察输出结果。

In [ ]:
y_pred = gs.best_estimator_.predict(X_test)
print(f"\n=== 预测值 vs 真实值（前 10 个）===")
print(f"测试 MSE: {mean_squared_error(y_test, y_pred):.4f}")
for i in range(10):
    print(f"  真实={y_test[i]:>8.2f}, 预测={y_pred[i]:>8.2f}, 残差={y_test[i]-y_pred[i]:>8.2f}")

print("\n=== 决策树回归要点 ===")
print("- 叶节点输出该节点所有训练样本的均值")
print("- 天然可处理非线性关系，不需要特征缩放")
print("- 极易过拟合（无限制深度时训练 R²=1.0）")
print("- 需要剪枝（限制 max_depth / min_samples_leaf）")
# --- 输出结果 ---
print("- 方差大：数据微小变化可能导致完全不同的树")

## 关键代码解释

### 不限深度 = 完美记忆

```python
dt = DecisionTreeRegressor(random_state=42)
dt.fit(X_train, y_train)
# 训练 R² = 1.0，每个叶子恰好包含一个训练样本
```

无限制的决策树会创建与训练样本数一样多的叶节点——完美拟合训练数据，但泛化能力为零。

### 剪枝参数

```python
for depth in [2, 3, 5, 8, 10, 15, None]:
```

关键参数：max_depth（最大深度）、min_samples_split（分裂最少样本数）、min_samples_leaf（叶节点最少样本数）。限制任一个都能防止过拟合。

## 使用示例

```python
from sklearn.tree import DecisionTreeRegressor
dt = DecisionTreeRegressor(max_depth=5, min_samples_leaf=5)
dt.fit(X_train, y_train)
print(dict(zip(feature_names, dt.feature_importances_)))
```

## 注意事项

1. **方差大**：数据微小变化可能导致完全不同的树
2. **外推能力差**：决策树不能预测训练数据范围之外的值
3. **不需要特征缩放**
4. **max_depth 是最重要的调参目标**

## 总结与延伸

以上代码展示了 **决策树 回归** 的完整流程。

**解读要点**：
- 关注 **R² 分数** 和 **MSE/MAE** 等回归指标
- R² 越接近 1 说明拟合越好
- 观察预测值 vs 真实值的散点图是否沿对角线分布

**进一步思考**：尝试修改关键参数（如正则化强度、学习率、树深度等），观察结果如何变化。

### 延伸阅读

- **回归树的分裂准则**：最小化均方误差（MSE）或平均绝对误差（MAE）
- **CART 回归树**：sklearn 使用 CART 算法，每次二分
- **树的集成**：随机森林、XGBoost、LightGBM 都是基于回归树的集成方法
